In [1]:

# Luego importamos otras librerias que seguramente vamos a necesitar

import pandas as pd
import numpy as np
import re
import unicodedata
#from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [2]:
df_train = pd.read_csv("train.csv")
data = df_train.copy()

## 1. Procesamiento de texto

Teniendo en cuenta que los textos crudos no son datos estructurados, se realiza NLP (natural languaje processing) para poder transformarlos en vectores y, de esta manera, permitir que los algoritmos de aprendizaje puedan usarlos.

Para esto, se utilizan representaciones dispersas (matrices dispersas) que permiten representar cada frase como un vector en un espacio vectorial semantico de dimensiones $\mathbb{R}^{|V|}$, donde |V| es el tamaño del vocabulario. Se hace uso del pipeline general definido para realizar este tipo de procesamiento: 
```text
Texto crudo
    ↓  Limpieza (lowercase, puntuación)
Texto normalizado
    ↓  Tokenización
Lista de tokens
    ↓  Stop words + Stemming
Tokens procesados
    ↓  Vectorización (BoW / TF-IDF)
Vector numérico ∈ ℝ^|V|
```

In [3]:
stop_words_es = set(stopwords.words('spanish'))

# Palabras relevantes que no queremos eliminar porque cambiarian la interpretación del modelo
palabras_relevantes = {"no", "nunca", "jamás", "sin", "ni", "tampoco",
"pero", "aunque","muy", "bastante", "demasiado", "más", "menos", "poco"}

stop_words_es -= palabras_relevantes

stemmer = SnowballStemmer('spanish')

def preprocesar_texto(texto):
    # A. Normalización: Convertir a minúsculas
    texto = texto.lower()
    
    # B. Normalización: Quitar signos de puntuación (deja solo letras y espacios)
    texto = re.sub(r'[^\w\s]', '', texto)
    
    # C. Tokenización: Separar por espacios
    tokens = texto.split()
    
    # D. Stopwords y Stemming: Filtrar palabras vacías y extraer la raíz
    tokens_procesados = [
        stemmer.stem(palabra) 
        for palabra in tokens 
        if palabra not in stop_words_es
    ]
    
    # E. Volver a unir en una sola cadena para el vectorizador
    return " ".join(tokens_procesados)

# 3. Aplicar el pipeline al DataFrame
df_train['text_limpio'] = df_train['text'].apply(preprocesar_texto)

# 4. Vectorizar (Bolsa de Palabras)
vec = CountVectorizer(
    max_features=10000, #Tope máximo de columnas (términos) que se van a generar, si hay más de 10.000 términos, se seleccionan los 10.000 más frecuentes
    min_df=3,     # Elimina términos o n-gramas que aparecen en menos de 3 reseñas    
    ngram_range=(1, 2)  # Considerar unigramas y bigramas, se cuentan tanto palabras indivuales como pares de palabras consecutivas
)

#Vectorización completa de los textos limpios, generando una matriz dispersa donde cada fila representa un texto y cada columna representa un término (palabra o n-grama) del vocabulario generado por el vectorizador. Los valores en la matriz indican la frecuencia de cada término en cada texto.
#Esto es lo que usamos para ya entrenar el modelo que hagamos
X_train = vec.fit_transform(df_train['text_limpio'])